# 03b — GNN v2: three-way scaffold split + edge-aware GINE

Phase 3.5 of **CRC_Inhibitor_ML**. Same task as Phase 3.0 (single-target pIC50 regression), but with four targeted fixes for the issues diagnosed in Phase 3.0:

| Fix | What it does | Phase 3.0 had |
|---|---|---|
| **Three-way scaffold split** | Train, val, AND test all contain disjoint scaffold groups — val now genuinely predicts test performance | Random val carved from scaffold-train, val and test had different scaffold distributions |
| **GINEConv** (edge-aware) | Uses bond features (type, conjugation, ring membership) during message passing | Used GINConv, which ignored bond features entirely |
| **BatchNorm between conv layers** | Stabilizes training, helps small-data convergence | No normalization |
| **ReduceLROnPlateau scheduler** | Halves learning rate when val plateaus, lets the model fine-tune | Single fixed LR for 200 epochs |

Architecture is otherwise identical to Phase 3.0 (3 message-passing layers, hidden_dim=128, dropout=0.2, batch size 64, Adam, early stopping with patience 20).

Output: `reports/gnn_v2_metrics.json`, model checkpoints at `models/gine_<target>.pt`. Cell 7 prints a three-way comparison: RF baseline vs Phase 3.0 GIN vs Phase 3.5 GINE.

In [ ]:
from pathlib import Path
from collections import defaultdict
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_add_pool

from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

RDLogger.DisableLog("rdApp.*")

PROJECT_ROOT  = Path(r"C:\Users\palla\OneDrive\Documents\Coding Projects\CRC_Inhibitor_ML")
CLEAN_CSV     = PROJECT_ROOT / "data" / "interim"   / "chembl_crc_targets_clean.csv"
MODELS_DIR    = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR   = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
GNN_V2_METRICS_PATH = REPORTS_DIR / "gnn_v2_metrics.json"
RF_METRICS_PATH     = REPORTS_DIR / "baseline_rf_metrics.json"
GNN_V1_METRICS_PATH = REPORTS_DIR / "gnn_single_target_metrics.json"

TARGETS = {
    "CHEMBL2189121": "KRAS",
    "CHEMBL5145":    "BRAF",
    "CHEMBL203":     "EGFR",
    "CHEMBL4005":    "PIK3CA",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## Cell 1 — load clean data + recompute scaffolds

Same as Phase 3.0. Self-contained notebook.

In [ ]:
clean = pd.read_csv(CLEAN_CSV)
print(f"Loaded {len(clean):,} rows")

print("\nParsing SMILES + computing scaffolds...")
mols = [Chem.MolFromSmiles(s) for s in tqdm(clean["canonical_smiles_std"])]
valid = np.array([m is not None for m in mols])
clean = clean[valid].reset_index(drop=True)
mols  = [m for m, ok in zip(mols, valid) if ok]

scaffolds = []
for m in tqdm(mols):
    try:
        scaffolds.append(MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False))
    except Exception:
        scaffolds.append("")
clean["scaffold"] = scaffolds

print(f"\nReady: {len(clean):,} molecules, {clean['scaffold'].nunique():,} unique scaffolds")

## Cell 2 — featurize as PyG molecular graphs

Identical to Phase 3.0 — 24 atom features per node, 6 bond features per edge. The only difference in Phase 3.5 is that the model will actually USE the edge features (via GINEConv) instead of ignoring them.

In [ ]:
ATOM_TYPES = ["C", "N", "O", "F", "P", "S", "Cl", "Br", "I", "B", "Si", "H", "Other"]
HYB_TYPES = [
    Chem.HybridizationType.S, Chem.HybridizationType.SP, Chem.HybridizationType.SP2,
    Chem.HybridizationType.SP3, Chem.HybridizationType.SP3D, Chem.HybridizationType.SP3D2,
]
BOND_TYPES = [
    Chem.BondType.SINGLE, Chem.BondType.DOUBLE, Chem.BondType.TRIPLE, Chem.BondType.AROMATIC,
]
N_ATOM_FEATURES = len(ATOM_TYPES) + len(HYB_TYPES) + 5
N_BOND_FEATURES = len(BOND_TYPES) + 2

def one_hot(value, options):
    return [int(value == o) for o in options]

def atom_features(atom):
    sym = atom.GetSymbol() if atom.GetSymbol() in ATOM_TYPES else "Other"
    return (
        one_hot(sym, ATOM_TYPES)
        + one_hot(atom.GetHybridization(), HYB_TYPES)
        + [
            atom.GetFormalCharge(),
            atom.GetTotalNumHs(),
            atom.GetDegree(),
            int(atom.GetIsAromatic()),
            int(atom.IsInRing()),
        ]
    )

def bond_features(bond):
    return (
        one_hot(bond.GetBondType(), BOND_TYPES)
        + [int(bond.GetIsConjugated()), int(bond.IsInRing())]
    )

def mol_to_pyg(mol, y_value):
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    src, dst, eattr = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        src += [i, j]
        dst += [j, i]
        eattr += [bf, bf]
    if len(src) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, N_BOND_FEATURES), dtype=torch.float)
    else:
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        edge_attr  = torch.tensor(eattr, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([y_value], dtype=torch.float))

print(f"N_ATOM_FEATURES = {N_ATOM_FEATURES}, N_BOND_FEATURES = {N_BOND_FEATURES}")
print("\nFeaturizing molecules...")
pyg_data = []
for mol, y in tqdm(list(zip(mols, clean["pic50"].values)), total=len(mols)):
    pyg_data.append(mol_to_pyg(mol, float(y)))
print(f"Built {len(pyg_data):,} graphs")

## Cell 3 — three-way scaffold split (THE KEY FIX)

Splits scaffolds into three disjoint buckets: ~70% train, ~10% val, ~20% test. Largest scaffold groups go to train first (most data signal), then val, then test.

Critically: **no scaffold appears in more than one split**. So val now answers the same question as test ("how well do we generalize to novel chemistries?"), and early-stopping on val will pick a model that actually generalizes.

Assertion at the end verifies the three splits are truly disjoint — catches any bug in the split logic immediately.

In [ ]:
def three_way_scaffold_split(scaffolds, frac_train=0.7, frac_val=0.1):
    """
    Three-way Bemis-Murcko scaffold split.
    All molecules sharing a scaffold go to the same split. Largest scaffold
    groups fill train, then val, then test. Deterministic given input order.
    """
    groups = defaultdict(list)
    for i, s in enumerate(scaffolds):
        groups[s].append(i)
    sorted_groups = sorted(groups.values(), key=len, reverse=True)

    n_total = len(scaffolds)
    train_cutoff = int(frac_train * n_total)
    val_cutoff   = train_cutoff + int(frac_val * n_total)

    train, val, test = [], [], []
    for indices in sorted_groups:
        if len(train) + len(indices) <= train_cutoff:
            train.extend(indices)
        elif len(train) + len(val) + len(indices) <= val_cutoff:
            val.extend(indices)
        else:
            test.extend(indices)
    return np.array(train), np.array(val), np.array(test)

def split_target(target_id):
    """Per-target three-way scaffold split."""
    mask = (clean["target_chembl_id"] == target_id).values
    target_indices = np.where(mask)[0]
    target_scaffolds = clean.loc[mask, "scaffold"].tolist()

    tr_local, va_local, te_local = three_way_scaffold_split(target_scaffolds)

    # Sanity check: scaffolds are disjoint across splits
    tr_scaff = set(target_scaffolds[i] for i in tr_local)
    va_scaff = set(target_scaffolds[i] for i in va_local)
    te_scaff = set(target_scaffolds[i] for i in te_local)
    assert tr_scaff.isdisjoint(va_scaff), "train and val share scaffolds!"
    assert tr_scaff.isdisjoint(te_scaff), "train and test share scaffolds!"
    assert va_scaff.isdisjoint(te_scaff), "val and test share scaffolds!"

    return (
        target_indices[tr_local],
        target_indices[va_local],
        target_indices[te_local],
    )

## Cell 4 — GINE model (edge-aware GIN)

`GINEConv` is the edge-feature-aware variant of GIN. The math: instead of just aggregating neighbor atom features `h_j`, it aggregates `ReLU(h_j + e_{ij})` — the neighbor's atom representation plus its bond's representation. So bond type, conjugation, and ring-membership all flow into the message passing.

Quirk: GINEConv requires edge features to be the same dimension as node features. We handle that with an `edge_proj` linear layer that maps the 6-dim bond features → 128-dim hidden representation, matching the projected atom features.

BatchNorm added between conv layers to stabilize training. Predictor head and dropout unchanged.

In [ ]:
class GINEMolecule(nn.Module):
    def __init__(self, in_dim, edge_dim, hidden_dim=128, n_layers=3, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_proj  = nn.Linear(edge_dim, hidden_dim)

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        self.dropout = nn.Dropout(dropout)
        self.predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        h = self.input_proj(x)
        e = self.edge_proj(edge_attr)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, edge_index, e)
            h = bn(h)
            h = F.relu(h)
            h = self.dropout(h)
        g = global_add_pool(h, batch)
        return self.predictor(g).squeeze(-1)

## Cell 5 — train loop with LR scheduler

Added `ReduceLROnPlateau` — when val RMSE stops improving for 10 epochs, halve the learning rate. Lets the model fine-tune in late epochs that Phase 3.0 wasted on too-high LR. Early stopping (patience 20) still kicks in if val RMSE truly plateaus.

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    preds_all, y_all = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch).cpu().numpy()
            preds_all.append(pred)
            y_all.append(batch.y.cpu().numpy())
    preds = np.concatenate(preds_all)
    y     = np.concatenate(y_all)
    return {
        "r2":   float(r2_score(y, preds)),
        "rmse": float(np.sqrt(mean_squared_error(y, preds))),
        "mae": float(mean_absolute_error(y, preds)),
    }

def train_target(train_data, val_data, test_data, target_name,
                 epochs=200, lr=1e-3, batch_size=64, patience=20):
    model = GINEMolecule(in_dim=N_ATOM_FEATURES, edge_dim=N_BOND_FEATURES).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=10
    )
    loss_fn = nn.MSELoss()

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_data,  batch_size=batch_size, shuffle=False)

    best_val_rmse = float("inf")
    best_state    = None
    epochs_no_improve = 0

    pbar = tqdm(range(epochs), desc=f"{target_name:7s}")
    for epoch in pbar:
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            pred = model(batch)
            loss = loss_fn(pred, batch.y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * batch.num_graphs
        train_loss = running_loss / len(train_data)

        val_metrics = evaluate_model(model, val_loader)
        scheduler.step(val_metrics["rmse"])

        if val_metrics["rmse"] < best_val_rmse:
            best_val_rmse = val_metrics["rmse"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        pbar.set_postfix({
            "train_mse": f"{train_loss:.3f}",
            "val_rmse":  f"{val_metrics['rmse']:.3f}",
            "best_val":  f"{best_val_rmse:.3f}",
            "lr":        f"{optimizer.param_groups[0]['lr']:.1e}",
        })

        if epochs_no_improve >= patience:
            break

    model.load_state_dict(best_state)
    test_metrics = evaluate_model(model, test_loader)
    test_metrics["n_train"]   = len(train_data)
    test_metrics["n_val"]     = len(val_data)
    test_metrics["n_test"]    = len(test_data)
    test_metrics["n_epochs"]  = epoch + 1
    test_metrics["best_val_rmse"] = float(best_val_rmse)
    return model, test_metrics

## Cell 6 — train per-target

Same loop as Phase 3.0, but uses the new split + new architecture. Saves checkpoints to `models/gine_<target>.pt`.

In [ ]:
all_results = []

for target_id, target_name in TARGETS.items():
    print(f"\n{'='*60}")
    print(f" {target_name} ({target_id})")
    print(f"{'='*60}")

    train_idx, val_idx, test_idx = split_target(target_id)
    train_data = [pyg_data[i] for i in train_idx]
    val_data   = [pyg_data[i] for i in val_idx]
    test_data  = [pyg_data[i] for i in test_idx]
    print(f"Train: {len(train_data):,}  Val: {len(val_data):,}  Test: {len(test_data):,}")

    if len(train_data) < 100 or len(val_data) < 20 or len(test_data) < 20:
        print(f"Skipping {target_name} — split too small.")
        continue

    model, metrics = train_target(train_data, val_data, test_data, target_name)
    metrics["target"] = target_name
    all_results.append(metrics)

    ckpt_path = MODELS_DIR / f"gine_{target_name.lower()}.pt"
    torch.save({
        "state_dict":       model.state_dict(),
        "in_dim":           N_ATOM_FEATURES,
        "edge_dim":         N_BOND_FEATURES,
        "target":           target_name,
        "target_chembl_id": target_id,
        "metrics":          metrics,
    }, ckpt_path)
    print(f"Final TEST: R\u00b2={metrics['r2']:.3f}  RMSE={metrics['rmse']:.3f}  MAE={metrics['mae']:.3f}")
    print(f"val\u2192test gap: {metrics['rmse'] - metrics['best_val_rmse']:+.3f} RMSE (Phase 3.0 was +0.2 to +0.4)")
    print(f"Saved checkpoint \u2192 {ckpt_path.name}")

results_df = pd.DataFrame(all_results)

## Cell 7 — three-way comparison: RF baseline vs Phase 3.0 GIN vs Phase 3.5 GINE

All three on the same metric (scaffold-split test R²) so deltas are meaningful. The story we want this table to tell:

1. The val→test gap should shrink dramatically (val now predicts test honestly)
2. Phase 3.5 R² should be > Phase 3.0 R²
3. Phase 3.5 R² should be ≥ RF baseline (the bar that matters)

In [ ]:
with open(RF_METRICS_PATH) as f:
    rf_results = json.load(f)
with open(GNN_V1_METRICS_PATH) as f:
    gnn_v1_results = json.load(f)

rf_df = pd.DataFrame(rf_results)
rf_scaffold = rf_df[rf_df["split"] == "scaffold"].set_index("target")[["r2", "rmse"]]
rf_scaffold.columns = ["r2_rf", "rmse_rf"]

gnn_v1_df = pd.DataFrame(gnn_v1_results).set_index("target")[["r2", "rmse"]]
gnn_v1_df.columns = ["r2_gnn_v1", "rmse_gnn_v1"]

gnn_v2_df = results_df.set_index("target")[["r2", "rmse"]]
gnn_v2_df.columns = ["r2_gnn_v2", "rmse_gnn_v2"]

compare = rf_scaffold.join(gnn_v1_df, how="inner").join(gnn_v2_df, how="inner")
compare["delta_v2_vs_rf"] = compare["r2_gnn_v2"] - compare["r2_rf"]
compare["delta_v2_vs_v1"] = compare["r2_gnn_v2"] - compare["r2_gnn_v1"]

compare = compare[[
    "r2_rf", "r2_gnn_v1", "r2_gnn_v2",
    "delta_v2_vs_rf", "delta_v2_vs_v1",
]]
print("Scaffold-split R² comparison (higher = better):\n")
print(compare.round(3))

## Cell 8 — save GNN v2 metrics

In [ ]:
with open(GNN_V2_METRICS_PATH, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"Saved {len(all_results)} GNN v2 metric records to {GNN_V2_METRICS_PATH}")
print()
print(json.dumps(all_results, indent=2))